# LLD-MMRI Liver Lesion PoC — GPU training runner
**Owner:** Roshani Pawar · **Repo:** `liver-lesion-mmri-poc`

Thin runner for Kaggle / Google Colab (T4 GPU).
All logic lives in `src/`; this notebook only:
1. Checks GPU and installs dependencies.
2. Clones the repo and downloads the dataset.
3. Builds manifest, splits, and ROI cache.
4. Smoke-tests the pipeline (< 5 min).
5. Trains all 12 configs with `--resume` support.
6. Evaluates and fills README placeholders.
7. Bundles reports for download.

**Prerequisites**
- Runtime: **GPU**. Colab: Runtime → Change runtime type → T4. Kaggle: Accelerator → GPU + Internet On.
- Set `REPO_URL` below.

**Licence:** LLD-MMRI is CC BY-NC 4.0 (research only, no redistribution).
Data is downloaded here for training and is never committed.

In [ ]:
# 1 — GPU check (hard stop if absent)
import torch, sys
print(f'PyTorch {torch.__version__}')
assert torch.cuda.is_available(), 'No CUDA GPU — fix runtime type before continuing.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
!nvidia-smi
!df -h / | tail -1

In [ ]:
# 2 — Configuration (EDIT THESE)
REPO_URL = 'https://github.com/RoshaniPawar16/liver-lesion-mmri-poc.git'  # EDIT if different
BRANCH   = 'main'

In [ ]:
# 3 — Clone repo + install CUDA-enabled PyTorch + dependencies
import os
if not os.path.exists('liver-lesion-mmri-poc'):
    !git clone --branch $BRANCH $REPO_URL
%cd liver-lesion-mmri-poc

!pip install -q torch==2.2.2 torchvision==0.17.2 \
    --index-url https://download.pytorch.org/whl/cu121
!pip install -q nibabel==5.2.1 scipy==1.13.0 scikit-learn==1.4.2 \
    pyyaml==6.0.1 matplotlib==3.8.4 tqdm==4.66.2

!git log --oneline -3   # record commit hash — goes into README

In [ ]:
# 4 — Download dataset (resumable; ~20 GB)
# If rate-limited: from huggingface_hub import login; login()
import os
if not os.path.exists('LLD-MMRI-MedSAM2/images'):
    !bash scripts/download_data.sh LLD-MMRI-MedSAM2
else:
    print('Dataset already present.')
!ls LLD-MMRI-MedSAM2/images | wc -l  # expect 3984

In [ ]:
# 5 — Manifest + splits (fast — always run to ensure up to date)
!python src/build_manifest.py --data-dir LLD-MMRI-MedSAM2 --output data/manifest.csv
!python src/check_splits.py --manifest data/manifest.csv --report reports/split_report.json

In [ ]:
# 6 — Build full ROI cache (~45 min on T4, ~1 GB output)
import pandas as pd
from pathlib import Path

idx = Path('data_cache/cache_index.csv')
if idx.exists() and len(pd.read_csv(idx)) == 3984:
    print(f'Full cache present ({len(pd.read_csv(idx))} tensors). Skipping.')
else:
    !python src/build_cache.py \
        --manifest data/manifest.csv \
        --data-dir LLD-MMRI-MedSAM2 \
        --cache-dir data_cache

In [ ]:
# 7 — Smoke test (verify pipeline end-to-end; must pass before full training)
!bash scripts/smoke_test.sh

In [ ]:
# 8 — Train Experiment A: phase ablation (headline results)
# --resume restarts from last checkpoint if the session died mid-run.
ABLATION_CONFIGS = [
    'configs/ablation_t2wi_dwi.yaml',
    'configs/ablation_pre_a.yaml',
    'configs/ablation_contrast_4phase.yaml',
    'configs/ablation_all_8.yaml',
]
for cfg in ABLATION_CONFIGS:
    print(f'\n=== {cfg} ===')
    !python src/train.py --config $cfg --resume

In [ ]:
# 9 — Train Experiment B: per-phase single-input (secondary table)
PER_PHASE_CONFIGS = [
    'configs/per_phase_C_pre.yaml',
    'configs/per_phase_C_A.yaml',
    'configs/per_phase_C_V.yaml',
    'configs/per_phase_C_Delay.yaml',
    'configs/per_phase_T2WI.yaml',
    'configs/per_phase_DWI.yaml',
    'configs/per_phase_InPhase.yaml',
    'configs/per_phase_OutPhase.yaml',
]
for cfg in PER_PHASE_CONFIGS:
    print(f'\n=== {cfg} ===')
    !python src/train.py --config $cfg --resume

In [ ]:
# 10 — Evaluate all runs (bootstrap CIs, calibration, confusion matrix)
!python src/evaluate.py --runs-dir outputs/runs --reports-dir reports
import pandas as pd
df = pd.read_csv('reports/results_summary.csv')
print(df[['run','n_phases','auroc','auroc_lo','auroc_hi','ece']].to_string(index=False))

In [ ]:
# 11 — Grad-CAM on best ablation run
import pandas as pd
summary = pd.read_csv('reports/results_summary.csv')
ablation = summary[summary['run'].str.startswith('ablation_')]
best_run = ablation.sort_values('auroc', ascending=False).iloc[0]['run']
print(f'Best ablation run: {best_run}')

!python src/gradcam.py \
    --run outputs/runs/$best_run \
    --manifest data/manifest.csv \
    --n-cases 6

In [ ]:
# 12 — Fill README.md placeholders from results_summary.csv
!python src/fill_readme.py \
    --summary reports/results_summary.csv \
    --readme README.md
print('README.md updated.')

In [ ]:
# 13 — Bundle reports + prediction CSVs for download
import shutil
from pathlib import Path

pred_out = Path('reports/predictions')
pred_out.mkdir(exist_ok=True)
for p in Path('outputs/runs').glob('*/predictions.csv'):
    shutil.copy(p, pred_out / f"{p.parent.name}_predictions.csv")

shutil.make_archive('results_bundle', 'zip', '.', 'reports')
print('results_bundle.zip ready.')

try:
    from google.colab import files
    files.download('results_bundle.zip')
except ImportError:
    print('Kaggle: download results_bundle.zip from the Output tab.')

## After this notebook
1. Unzip `results_bundle.zip` into the local repo (`reports/`, `outputs/.../predictions.csv`).
2. The README placeholders are already filled if cell 12 ran.
3. Record in progress.txt: GPU model, wall-clock per experiment, git commit hash (cell 3).
4. Commit, push, make the repo public.

**Session-death recovery:** re-run cells 1–6 (idempotent), then re-run cells 8–9 with `--resume` — training continues from the last saved checkpoint.